# 01 Create sample points  (requires the raw municipal data)

Draws the training and natural-test points from the building footprints (Frantz et al. 2021 label: area-weighted mean
height of the building parts whose centroid lies within 50 m), adds the five new cities, and rebalances the training set
towards tall buildings.

**This notebook needs the footprints, nDSMs and boundaries in `data_raw/`**, which are not in the repository (see
`data/SOURCES.md`). If you only want to reproduce the models, skip to notebook 04: the resulting points are already in the
shipped features table, and `scripts/make_gee_point_assets.py` writes the point files for Earth Engine from it.

Order: 1. sampler for the three first cities, 2. natural test points for them, 3. add the five new cities,
4. rebalance, 5. optional check of the height statistic of each provider.

In [ ]:
# Setup: make `config.py` importable whether Jupyter was started in the repo root or in notebooks/
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, _here / "notebooks", _here.parent / "notebooks"):
    if (_p / "config.py").exists():
        sys.path.insert(0, str(_p)); break
else:
    raise FileNotFoundError("config.py not found: start Jupyter from the repository root or from notebooks/")
import config
print("repo root:", config.ROOT)

## 1. Sampler (functions)

In [ ]:
#!/usr/bin/env python3
"""
generate_sample_points.py   (v3, follows Frantz et al. 2021, Sec. 4.3)

Builds the training/validation sample points the way the paper describes,
using your building footprints (one polygon per building "part" with a
height attribute, e.g. the LiDAR-derived heights for Porto Alegre).

Frantz et al.                         This script
-----------------------------------   -----------------------------------
10 m EO pixel grid                    10 m grid in a metric CRS (pixel
                                      edges on multiples of 10 m, like the
                                      Sentinel-2 UTM grid)
Pixel is a candidate if a building    same rule (centroid within 5 m of
part centroid lies within 5 m         the pixel centre)
Label = area-weighted mean height     same (Eq. 2): all parts whose
of all parts with centroid within     centroid is within 50 m, weights =
50 m of the pixel (buildings only,    footprint areas, ground is NOT
ground is not averaged in)            averaged in
Constrained stratification on that    1 m slices from 3 m upward; the
label, 1 m slices, same quota per     last slice [49 m, inf) pools the
slice, slices visited in alternating  rare high-rises; alternating order
order (3-4, 49-50, 4-5, 48-49, ...)   low, high, low, high...
Candidates within 50 m of an          same (discarded, never revisited)
already accepted sample are dropped
Zero-height background: Poisson-disc  random dart throwing + minimum
points, removed if closer than 100 m  spacing, removed if closer than
to any building part centroid         100 m to any part centroid, and
                                      only where the nDSM has valid data
                                      (so missing LiDAR is never labelled
                                      "no building")

Differences to keep in mind (state them in the paper):
  * building parts = your footprint polygons (not CityGML LoD2 parts);
  * heights come from your footprint attribute (LiDAR-derived);
  * quota per slice is scaled down (Frantz: 1000 per slice for a whole
    federal state or city; here SAMPLES_PER_SLICE, default 200);
  * background points are random dart throwing with a minimum spacing, not
    the Poisson disc algorithm of Tulleken (2008).

Outputs per city (path set by `output` in CITIES):
  <output>.gpkg          projected CRS, for QGIS
  <output>.geojson       EPSG:4326, for uploading to Earth Engine as a table
                         asset (Earth Engine table uploads take Shapefile,
                         CSV, GeoJSON and TFRecord as far as I know; not
                         GeoPackage)
  <output>_wgs84.csv     same points as a CSV with longitude/latitude columns,
                         a fallback if the GeoJSON upload is refused
  <output>_slices.csv    per 1 m slice: candidate pixels in the whole city
                         (the "population"), samples drawn, quota. The
                         candidate counts are a stand-in for the reference
                         frequencies Frantz used to weight the RMSE; Frantz
                         took them from the full reference dataset, so label
                         this a deviation if you use it.
Properties: sample_id, city, height_m (label), is_background, n_parts_50m,
slice_lo, x_m, y_m (pixel centre, metric CRS)

If your footprints have no height attribute for a city, add one in QGIS
first (Zonal statistics of the nDSM over the footprints, e.g. the 90th
percentile) -- QGIS is much faster than doing it here.

Requires: numpy, scipy, geopandas, shapely, rasterio
Usage:
    python generate_sample_points.py --selftest        # synthetic check
    python generate_sample_points.py --list            # configured cities
    python generate_sample_points.py                   # ACTIVE_CITIES
    python generate_sample_points.py --cities sao_paulo
    python generate_sample_points.py --cities all --quota 100
"""

import argparse
import sys
import os
import tempfile
 
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from scipy.spatial import cKDTree
from shapely.geometry import Point

In [ ]:
import os
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

# gee_epsg: the CRS used in export_frantz_features_gee.* for this city. The
# sampler snaps points to a 10 m grid in the footprints' CRS; if that is not
# the same UTM zone the script warns.
_CITY_EPSG = {
    "porto_alegre": 32722, "sao_paulo": 32723, "rio_de_janeiro": 32723,
    "niteroi": 32723, "curitiba": 32722, "belo_horizonte": 32723,
    "salvador": 32724, "recife": 32725, "florianopolis": 32722,
}


CITIES = {
    name: dict(
        footprints=config.FOOTPRINTS(name),
        height_field="height_m",
        boundary=config.BOUNDARY(name),
        ndsm=config.NDSM(name),     # only used to know where LiDAR data exist
        output=os.path.join(config.OUT, "sample_points", name, "sample_points"),
        gee_epsg=epsg,
    )
    for name, epsg in _CITY_EPSG.items()
}

# Cities run by default. The others are configured with the same path pattern;
# edit their entries when their data are ready, then use --cities.
ACTIVE_CITIES = ["porto_alegre", "sao_paulo", "rio_de_janeiro"]

PIXEL_M = 10.0
CENTROID_TOL_M = 5.0          # candidate pixel: a part centroid within 5 m
NEIGHBOUR_RADIUS_M = 50.0     # parts used for the label
MIN_SAMPLE_DIST_M = 50.0      # spacing between accepted building samples
BG_MIN_DIST_TO_BUILDING_M = 100.0
BG_SPACING_M = 50.0           # minimum spacing between background samples
HEIGHT_MIN_M = 3              # first slice [3, 4)
HEIGHT_TOP_M = 49             # last slice [49, inf)
SAMPLES_PER_SLICE = 200
N_BACKGROUND = 200
MIN_PART_AREA_M2 = 10.0       # ignore tiny sheds/artefacts
MAX_ROUNDS = 100_000
SEED = 42

In [ ]:
# ---------------------------------------------------------------------
# Building parts
# ---------------------------------------------------------------------
 
def load_parts(path, height_field, min_area):
    gdf = gpd.read_file(path)
    if height_field not in gdf.columns:
        raise ValueError(
            f"{path} has no '{height_field}' field. Add building heights first "
            "(QGIS: Zonal statistics of the nDSM over the footprints).")
    if gdf.crs is None:
        raise ValueError(f"{path} has no CRS defined.")
    if gdf.crs.is_geographic:
        gdf = gdf.to_crs(gdf.estimate_utm_crs())
    n_in = len(gdf)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].explode(index_parts=False)
    gdf = gdf[gdf.geometry.geom_type == "Polygon"]
    gdf = gdf.assign(_z=pd.to_numeric(gdf[height_field], errors="coerce"), _a=gdf.geometry.area)
    gdf = gdf[(gdf["_z"] > 0) & np.isfinite(gdf["_z"]) & (gdf["_a"] >= min_area)]
    print(f"  {n_in} features read, {len(gdf)} kept (height > 0, area >= {min_area:g} m2)")
    cen = gdf.geometry.centroid
    return (cen.x.to_numpy(), cen.y.to_numpy(), gdf["_a"].to_numpy(), gdf["_z"].to_numpy(), gdf.crs)
 
 
def pixel_centre(v):
    return np.floor(v / PIXEL_M) * PIXEL_M + PIXEL_M / 2.0
 
 
def candidate_pixels(cx, cy):
    """Pixels whose centre is within CENTROID_TOL_M of a part centroid."""
    px, py = pixel_centre(cx), pixel_centre(cy)
    keep = np.hypot(px - cx, py - cy) <= CENTROID_TOL_M
    centres = np.unique(np.c_[px[keep], py[keep]], axis=0)
    return centres
 
 
def area_weighted_height(centres, tree, z, a, radius, chunk=20000):
    """Eq. 2 of the paper: sum(z_p * A_p) / sum(A_p) over parts whose
    centroid is within `radius` of the pixel centre."""
    zbar = np.empty(len(centres))
    n_parts = np.empty(len(centres), dtype=int)
    for s in range(0, len(centres), chunk):
        lists = tree.query_ball_point(centres[s:s + chunk], r=radius)
        for k, idx in enumerate(lists):
            idx = np.asarray(idx)
            w = a[idx]
            zbar[s + k] = (z[idx] * w).sum() / w.sum()
            n_parts[s + k] = len(idx)
    return zbar, n_parts
 
 
# ---------------------------------------------------------------------
# Sampling
# ---------------------------------------------------------------------
 
class SpacingIndex:
    """Grid hash for fast 'is there an accepted point closer than d?'."""
 
    def __init__(self, cell):
        self.cell, self.cells = cell, {}
 
    def _key(self, x, y):
        return int(x // self.cell), int(y // self.cell)
 
    def too_close(self, x, y, d):
        ci, cj = self._key(x, y)
        for di in (-1, 0, 1):
            for dj in (-1, 0, 1):
                for px, py in self.cells.get((ci + di, cj + dj), ()):
                    if (px - x) ** 2 + (py - y) ** 2 < d * d:
                        return True
        return False
 
    def add(self, x, y):
        self.cells.setdefault(self._key(x, y), []).append((x, y))
 
 
def make_slices():
    slices = [(lo, lo + 1) for lo in range(HEIGHT_MIN_M, HEIGHT_TOP_M)]
    slices.append((HEIGHT_TOP_M, np.inf))
    return slices
 
 
def alternating_order(n):
    lo, hi, order = 0, n - 1, []
    while lo <= hi:
        order.append(lo)
        if hi != lo:
            order.append(hi)
        lo, hi = lo + 1, hi - 1
    return order
 
 
def stratified_sample(zbar, centres, slices, quota, min_dist, rng):
    queues = []
    for lo, hi in slices:
        idx = np.where((zbar >= lo) & (zbar < hi))[0]
        rng.shuffle(idx)
        queues.append(idx)
    ptr = [0] * len(slices)
    counts = [0] * len(slices)
    index = SpacingIndex(min_dist)
    chosen = []
    order = alternating_order(len(slices))
 
    for _ in range(MAX_ROUNDS):
        active = False
        for s in order:
            if counts[s] >= quota or ptr[s] >= len(queues[s]):
                continue
            active = True
            while ptr[s] < len(queues[s]):
                c = queues[s][ptr[s]]
                ptr[s] += 1
                x, y = centres[c]
                if not index.too_close(x, y, min_dist):
                    index.add(x, y)
                    chosen.append((c, s))
                    counts[s] += 1
                    break
        if not active:
            break
    return chosen, counts, [len(q) for q in queues]
 
 
def valid_in_raster(path, xy, xy_crs):
    """True where the raster has data at the given metric coordinates."""
    with rasterio.open(path) as src:
        pts = gpd.GeoSeries([Point(p) for p in xy], crs=xy_crs)
        if src.crs is not None and src.crs != xy_crs:
            pts = pts.to_crs(src.crs)
        coords = np.c_[pts.x, pts.y]
        b = src.bounds
        inside = ((coords[:, 0] >= b.left) & (coords[:, 0] < b.right)
                  & (coords[:, 1] > b.bottom) & (coords[:, 1] <= b.top))
        ok = np.zeros(len(coords), dtype=bool)
        if inside.any():
            vals = np.array([v[0] for v in src.sample(map(tuple, coords[inside]))], dtype=float)
            good = np.isfinite(vals)
            if src.nodata is not None:
                good &= vals != src.nodata
            ok[np.where(inside)[0]] = good
    return ok
 
 
_BOUNDARY_CACHE = {}


def _eroded_boundary(path, xy_crs, inset_m):
    """Boundary polygon shrunk by inset_m, read once per run and repaired if needed."""
    import shapely
    key = (path, str(xy_crs), inset_m)
    if key not in _BOUNDARY_CACHE:
        b = gpd.read_file(path).to_crs(xy_crs)
        n_bad = int((b.geometry.isna() | b.geometry.is_empty).sum())
        b = b[b.geometry.notna() & ~b.geometry.is_empty]
        n_invalid = int((~b.geometry.is_valid).sum())
        # repair each geometry BEFORE the union: an invalid one makes the union itself fail
        geom = shapely.force_2d(shapely.union_all(shapely.make_valid(np.asarray(b.geometry.array))))
        shrunk = geom.buffer(-inset_m)
        print(f"  boundary file: {len(b)} features ({n_bad} empty dropped, {n_invalid} invalid repaired), "
              f"area {geom.area / 1e6:.1f} km2, {geom.area and shrunk.area / geom.area * 100:.1f}% left "
              f"after a {inset_m:.0f} m inset")
        if shrunk.is_empty:
            raise ValueError(f"{path}: nothing left after shrinking by {inset_m} m; "
                             "check the boundary's CRS and that it is a polygon layer")
        _BOUNDARY_CACHE[key] = shrunk
    return _BOUNDARY_CACHE[key]


def inside_boundary(path, xy, xy_crs, inset_m=NEIGHBOUR_RADIUS_M):
    """True where a point lies inside the boundary polygon shrunk by inset_m, so the
    whole 50 m label neighbourhood is inside the area with LiDAR and complete footprints.
    The nDSMs are rectangles around the city, so their extent cannot tell us this."""
    import shapely
    return shapely.contains_xy(_eroded_boundary(path, xy_crs, inset_m), xy[:, 0], xy[:, 1])


def background_points(cx, cy, tree, ndsm_path, crs, n_wanted, rng, boundary=None):
    xmin, xmax, ymin, ymax = cx.min(), cx.max(), cy.min(), cy.max()
    n_try = max(n_wanted * 100, 20000)
    cand = np.c_[rng.uniform(xmin, xmax, n_try), rng.uniform(ymin, ymax, n_try)]
    cand = np.unique(np.c_[pixel_centre(cand[:, 0]), pixel_centre(cand[:, 1])], axis=0)
    rng.shuffle(cand)
 
    d, _ = tree.query(cand)
    cand = cand[d >= BG_MIN_DIST_TO_BUILDING_M]
    if ndsm_path and len(cand):
        cand = cand[valid_in_raster(ndsm_path, cand, crs)]
    if boundary and len(cand):
        cand = cand[inside_boundary(boundary, cand, crs)]
 
    index, kept = SpacingIndex(BG_SPACING_M), []
    for x, y in cand:
        if len(kept) >= n_wanted:
            break
        if not index.too_close(x, y, BG_SPACING_M):
            index.add(x, y)
            kept.append((x, y))
    if len(kept) < n_wanted:
        print(f"  WARNING: only {len(kept)} background points found (wanted {n_wanted}); "
              "the footprint layer may leave little open space in the LiDAR area")
    return np.array(kept).reshape(-1, 2)
 
 
# ---------------------------------------------------------------------
# Per city
# ---------------------------------------------------------------------
 
def run_city(name, cfg, quota=SAMPLES_PER_SLICE, n_bg=N_BACKGROUND):
    rng = np.random.default_rng(SEED)
    print(f"{name}:")
    boundary = cfg.get("boundary")           # checked first: loading the footprints can take minutes
    if not boundary or not os.path.exists(boundary):
        raise FileNotFoundError(f"{name}: boundary file not found: {boundary}")
    cx, cy, area, z, crs = load_parts(cfg["footprints"], cfg["height_field"], MIN_PART_AREA_M2)
    epsg = crs.to_epsg()
    print(f"  {len(cx)} building parts, CRS {crs.to_string()}")
    want = cfg.get("gee_epsg")
    if want and epsg != want:
        print(f"  NOTE: footprints are in EPSG:{epsg}, the Earth Engine config uses EPSG:{want}. "
              "Different realizations of the same UTM zone (e.g. SIRGAS 2000 vs WGS 84) differ by "
              "well under a pixel and are fine; a different zone is not, so check the zone.")
    tree = cKDTree(np.c_[cx, cy])
 
    centres = candidate_pixels(cx, cy)
    keep = inside_boundary(boundary, centres, crs)
    print(f"  boundary: {int((~keep).sum())} of {len(centres)} candidate pixels are outside the boundary "
          f"or within {NEIGHBOUR_RADIUS_M:.0f} m of its edge, removed")
    centres = centres[keep]
    if len(centres) == 0:
        raise ValueError(f"{name}: no candidate pixel inside the boundary; check that the boundary "
                         "and the footprints cover the same place and CRS")
    zbar, n_parts = area_weighted_height(centres, tree, z, area, NEIGHBOUR_RADIUS_M)
    print(f"  {len(centres)} candidate pixels, label range {zbar.min():.1f}-{zbar.max():.1f} m")
 
    slices = make_slices()
    chosen, counts, pop = stratified_sample(zbar, centres, slices, quota, MIN_SAMPLE_DIST_M, rng)
    short = [f"{lo}-{'inf' if np.isinf(hi) else hi}m:{c}" for (lo, hi), c in zip(slices, counts) if c < quota]
    print(f"  {len(chosen)} building samples; slices below quota: {', '.join(short) or 'none'}")
 
    bg = background_points(cx, cy, tree, cfg.get("ndsm"), crs, n_bg, rng, boundary)
 
    rows = []
    for c, s in chosen:
        rows.append(dict(x_m=centres[c, 0], y_m=centres[c, 1], height_m=float(zbar[c]),
                         is_background=0, n_parts_50m=int(n_parts[c]), slice_lo=slices[s][0]))
    for x, y in bg:
        rows.append(dict(x_m=x, y_m=y, height_m=0.0, is_background=1, n_parts_50m=0, slice_lo=-1))
 
    gdf = gpd.GeoDataFrame(rows, geometry=[Point(r["x_m"], r["y_m"]) for r in rows], crs=crs)
    gdf.insert(0, "city", name)
    gdf.insert(0, "sample_id", [f"{name}_{i:06d}" for i in range(len(gdf))])
    out = cfg["output"]
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    gdf.to_file(out + ".gpkg", driver="GPKG")
    wgs = gdf.to_crs(4326)
    wgs.to_file(out + ".geojson", driver="GeoJSON")
    csv = pd.DataFrame(wgs.drop(columns="geometry"))
    csv.insert(len(csv.columns), "longitude", wgs.geometry.x.to_numpy())
    csv.insert(len(csv.columns), "latitude", wgs.geometry.y.to_numpy())
    csv.to_csv(out + "_wgs84.csv", index=False)
    pd.DataFrame({
        "slice_lo": [lo for lo, _ in slices],
        "slice_hi": [("" if np.isinf(hi) else hi) for _, hi in slices],
        "candidate_pixels": pop,
        "samples": counts,
        "quota": quota,
    }).to_csv(out + "_slices.csv", index=False)
    print(f"  wrote {len(gdf)} points ({len(bg)} background) -> {out}.gpkg / .geojson / _wgs84.csv / _slices.csv")
    return gdf, (cx, cy, area, z)
 
 
# ---------------------------------------------------------------------
# Self-test on synthetic data
# ---------------------------------------------------------------------
 
def selftest():
    from rasterio.transform import from_origin
    from shapely.geometry import box
 
    rng = np.random.default_rng(1)
    d = tempfile.mkdtemp()
    x0, y0, size = 300_000.0, 6_700_000.0, 6000.0
    polys, heights = [], []
    for _ in range(7000):
        x, y = rng.uniform(x0, x0 + size), rng.uniform(y0, y0 + size)
        if x0 + 2500 < x < x0 + 3500 and y0 + 2500 < y < y0 + 3500:
            continue  # an empty "park", so background points can exist
        w, h = rng.uniform(8, 30, 2)
        polys.append(box(x, y, x + w, y + h))
        heights.append(float(np.clip(rng.lognormal(2.2, 0.7), 2, 90)))
    gpd.GeoDataFrame({"height_m": heights}, geometry=polys, crs=32722).to_file(
        os.path.join(d, "fp.gpkg"), driver="GPKG")
 
    n = int(size / 10)
    arr = np.ones((n, n), dtype="float32")
    arr[:, :60] = np.nan   # western strip without LiDAR
    with rasterio.open(os.path.join(d, "ndsm.tif"), "w", driver="GTiff", height=n, width=n, count=1,
                       dtype="float32", crs="EPSG:32722", nodata=np.nan,
                       transform=from_origin(x0, y0 + size, 10, 10)) as dst:
        dst.write(arr, 1)
 
    bnd = box(x0 + 1000, y0 + 500, x0 + 5000, y0 + 5500)      # the "city": smaller than the footprints
    gpd.GeoDataFrame(geometry=[bnd], crs=32722).to_file(os.path.join(d, "boundary.gpkg"), driver="GPKG")

    cfg = dict(footprints=os.path.join(d, "fp.gpkg"), height_field="height_m",
               ndsm=os.path.join(d, "ndsm.tif"), boundary=os.path.join(d, "boundary.gpkg"),
               output=os.path.join(d, "out", "pts"),
               gee_epsg=32723)       # deliberately a different zone, to exercise the NOTE
    gdf, (cx, cy, area, z) = run_city("synthetic", cfg, quota=15, n_bg=40)
 
    bld = gdf[gdf.is_background == 0]
    bgd = gdf[gdf.is_background == 1]
    xy = np.c_[bld.x_m, bld.y_m]
    dd = np.hypot(xy[:, None, 0] - xy[None, :, 0], xy[:, None, 1] - xy[None, :, 1])
    np.fill_diagonal(dd, np.inf)
    assert dd.min() >= MIN_SAMPLE_DIST_M, "building samples closer than 50 m"
 
    for _, r in bld.sample(5, random_state=0).iterrows():       # recompute Eq. 2 by hand
        m = np.hypot(cx - r.x_m, cy - r.y_m) <= NEIGHBOUR_RADIUS_M
        assert abs((z[m] * area[m]).sum() / area[m].sum() - r.height_m) < 1e-6, "label mismatch"
    assert (bld.height_m >= HEIGHT_MIN_M).all(), "building label below 3 m"
 
    bxy = np.c_[bgd.x_m, bgd.y_m]
    dmin = np.hypot(bxy[:, None, 0] - cx[None, :], bxy[:, None, 1] - cy[None, :]).min(axis=1)
    assert dmin.min() >= BG_MIN_DIST_TO_BUILDING_M, "background too close to a building"
    assert (bgd.x_m > x0 + 600).all(), "background fell into the no-LiDAR strip"
    assert (bgd.height_m == 0).all()
    import shapely
    inner = bnd.buffer(-NEIGHBOUR_RADIUS_M)
    assert shapely.contains_xy(inner, gdf.x_m.to_numpy(), gdf.y_m.to_numpy()).all(), \
        "a point lies outside the boundary shrunk by 50 m"
    assert not shapely.contains_xy(inner, cx, cy).all(), "test boundary removed nothing"
    per_slice = bld.groupby("slice_lo").size()
    assert per_slice.max() <= 15, "quota exceeded"
 
    out = cfg["output"]
    sl = pd.read_csv(out + "_slices.csv")
    assert len(sl) == len(make_slices()) and (sl.samples <= sl.quota).all(), "slices csv wrong"
    assert (sl.samples <= sl.candidate_pixels).all(), "more samples than candidates"
    assert sl.samples.sum() == len(bld), "slices csv does not add up to the building samples"
    wc = pd.read_csv(out + "_wgs84.csv")
    assert len(wc) == len(gdf) and wc.longitude.between(-180, 180).all() and wc.latitude.between(-90, 90).all()
    gj = gpd.read_file(out + ".geojson")
    assert len(gj) == len(gdf) and gj.crs.to_epsg() == 4326, "geojson wrong"
    need = {"sample_id", "city", "height_m", "is_background", "x_m", "y_m"}   # columns the GEE script selects
    assert need <= set(gj.columns), f"geojson missing {need - set(gj.columns)}"
    print("\nSELFTEST PASSED: spacing >= 50 m, labels match Eq. 2, background >= 100 m from "
          f"buildings and inside LiDAR coverage and the boundary, quota respected, slices/CSV/GeoJSON outputs consistent "
          f"({len(bld)} building + {len(bgd)} background samples)")
 
 
def _parse(ap, argv=None):
    """Strict in a terminal. Inside Jupyter, ignore the kernel's own
    '-f <connection file>' arguments instead of exiting with an error."""
    if argv is None and "ipykernel" in sys.modules:
        return ap.parse_known_args(sys.argv[1:])[0]
    return ap.parse_args(argv)
 
 
def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--selftest", action="store_true")
    ap.add_argument("--list", action="store_true", help="show configured cities and exit")
    ap.add_argument("--cities", default=None,
                    help="comma list of city keys, or 'all' (default: ACTIVE_CITIES)")
    ap.add_argument("--quota", type=int, default=SAMPLES_PER_SLICE)
    ap.add_argument("--background", type=int, default=N_BACKGROUND)
    args = _parse(ap, argv)
    if args.selftest:
        selftest()
        return
    if args.list:
        for k, v in CITIES.items():
            flag = "active" if k in ACTIVE_CITIES else "      "
            print(f"{flag}  {k:15s} {v['footprints']}  (exists: {os.path.exists(v['footprints'])})")
            print(f"{'':22s}boundary: {v.get('boundary')}  (exists: {os.path.exists(v.get('boundary') or '')})")
        return
    if args.cities is None:
        names = ACTIVE_CITIES
    elif args.cities == "all":
        names = list(CITIES)
    else:
        names = [c.strip() for c in args.cities.split(",") if c.strip()]
    unknown = [c for c in names if c not in CITIES]
    if unknown:
        raise SystemExit(f"Unknown cities {unknown}. Valid: {list(CITIES)}")
    for name in names:
        run_city(name, CITIES[name], quota=args.quota, n_bg=args.background)
 

## 2. Training points of the three first cities
Writes `outputs/sample_points/<city>/sample_points*` (GPKG, GeoJSON, WGS84 CSV). Upload the WGS84 file to Earth Engine
as the asset `<city>_sample_points_selected_wgs84` (see `gee/export_frantz_features.js`).

In [ ]:
main(["--cities", ",".join(ACTIVE_CITIES)])

## 3. Reference table for the next steps
Rebuilds, from the shipped 8-city table, the 3-city table as it was before the rebalancing, and the training rows used to
keep natural test points 50 m away from training points.

In [ ]:
# The three first cities (Porto Alegre, Sao Paulo, Rio) were sampled BEFORE the rebalancing. The shipped 8-city table
# already has the rebalancing applied: rows the rebalancing dropped carry split == "unused". Undo that to get the
# "current merged table" the sampler and the rebalancing code below expect (train + unused -> train).
import pandas as pd
OUT_NEW = config.out("new_cities")
full = config.read_table(usecols=["sample_id", "city", "height_m", "is_background", "x_m", "y_m", "split"])
old3 = full[full.city.isin(["porto_alegre", "sao_paulo", "rio_de_janeiro"])].copy()
old3["split"] = old3["split"].replace({"unused": "train"})
THREE_CITY_TABLE = os.path.join(OUT_NEW, "three_city_table_before_rebalancing.csv")
old3.to_csv(THREE_CITY_TABLE, index=False)
THREE_CITY_TRAIN_ROWS = os.path.join(OUT_NEW, "three_city_training_rows.csv")
old3[old3.split != "test"].to_csv(THREE_CITY_TRAIN_ROWS, index=False)    # reference for the 50 m exclusion of the natural test points
print(old3.groupby(["city", "split"]).size().unstack(fill_value=0))

## 4. Natural test points of the three first cities

In [ ]:
# ---------------------------------------------------------------------
# Natural-distribution TEST points (deviation from Frantz, see notes below)
# Runs in the same notebook as the sampler: it reuses load_parts, candidate_pixels,
# area_weighted_height, inside_boundary, background_points, SpacingIndex and CITIES.
#
# The test points are drawn at random from ALL candidate pixels (no height
# stratification), so their height distribution is the city's own. Independence from
# the training points is enforced in one of two ways:
#   drop_train=False (default): test points are only taken from pixels at least
#       buffer_m from every training point. The training set stays whole, but the
#       buffer removes tall buildings from the test pool (the training sample already
#       sits on nearly all of them), so the test set is no longer natural.
#   drop_train=True: test points are drawn from ALL candidate pixels, then every
#       training point within buffer_m of a test point is flagged for removal
#       (written to <output>_natural_drop_train.csv; nothing is deleted). The test
#       set is exactly natural; the cost is a few training points.
# Either way this is a distance rule, not a spatial block hold-out.
# ---------------------------------------------------------------------

NAT_CLASSES = [(-np.inf, 3, "<3 m"), (3, 6, "3-6 m"), (6, 15, "6-15 m"),
               (15, 30, "15-30 m"), (30, np.inf, ">30 m")]


def _class_counts(z):
    z = np.asarray(z)
    return [int(((z >= lo) & (z < hi)).sum()) for lo, hi, _ in NAT_CLASSES]


def natural_test_points(name, cfg, n_nat=1000, n_bg=100, buffer_m=500.0,
                        min_label_m=HEIGHT_MIN_M, seed=SEED + 1, dry_run=False, square_buffer=False,
                        train_table=None, drop_train=False,
                        scan=(25, 50, 75, 100, 200, 330, 500)):
    """Draw natural test points for one city.

    dry_run=True: only print how many candidate pixels survive each buffer, per height
    class, and write nothing.
    train_table: path of the merged feature table (or any CSV with city, x_m, y_m; with
    drop_train also sample_id, height_m, is_background). The exclusion then uses exactly the
    points that were exported, not the sampler's .gpkg, which a sampler rerun can overwrite.
    square_buffer=False: buffer is a radius (Euclidean distance), right for the table features
    (50 m windows: 100 m keeps both the label and feature windows apart).
    square_buffer=True: buffer is a square (max of |dx|, |dy|). Two square patches of width W
    overlap unless max(|dx|, |dy|) >= W, so use this with buffer_m = 330 for the CNN patches.
    drop_train=True: see the note at the top; needs train_table."""
    rng = np.random.default_rng(seed)
    print(f"{name} (natural test points):")
    boundary = cfg.get("boundary")
    if not boundary or not os.path.exists(boundary):
        raise FileNotFoundError(f"{name}: boundary file not found: {boundary}")
    if drop_train and not train_table:
        raise ValueError("drop_train=True needs train_table (sample_id, height_m, is_background, x_m, y_m)")
    train_path = cfg["output"] + ".gpkg"
    if not train_table and not os.path.exists(train_path):
        raise FileNotFoundError(f"{name}: existing sample points not found: {train_path}")

    cx, cy, area, z, crs = load_parts(cfg["footprints"], cfg["height_field"], MIN_PART_AREA_M2)
    tree = cKDTree(np.c_[cx, cy])
    tmeta = None
    if train_table:
        need = ["city", "x_m", "y_m"] + (["sample_id", "height_m", "is_background"] if drop_train else [])
        tt = pd.read_csv(train_table, usecols=need)
        tt = tt[tt.city == name].reset_index(drop=True)
        if tt.empty:
            raise ValueError(f"{name}: no rows for this city in {train_table}")
        txy = tt[["x_m", "y_m"]].to_numpy(float)
        tmeta = tt
        src = os.path.basename(train_table)
    else:
        train = gpd.read_file(train_path).to_crs(crs)
        txy = np.c_[train.geometry.x.to_numpy(), train.geometry.y.to_numpy()]
        src = os.path.basename(train_path)
    ttree = cKDTree(txy)
    print(f"  {len(txy)} existing points from {src} (they stay as they are)")

    centres = candidate_pixels(cx, cy)
    centres = centres[inside_boundary(boundary, centres, crs)]
    zbar, n_parts = area_weighted_height(centres, tree, z, area, NEIGHBOUR_RADIUS_M)
    pnorm = np.inf if square_buffer else 2
    d_train, _ = ttree.query(centres, p=pnorm)

    # how much of each height class is left for different buffers
    print("  candidate pixels left by buffer (label >= %.1f m):" % min_label_m)
    print("    buffer".ljust(12) + "".join(lbl.rjust(10) for _, _, lbl in NAT_CLASSES) + "     total")
    lab_ok = zbar >= min_label_m
    for b in [0] + list(scan):
        keep = lab_ok & (d_train >= b)
        cc = _class_counts(zbar[keep])
        print(f"    {b:>5} m  ".ljust(12) + "".join(f"{c:>10d}" for c in cc) + f"{int(keep.sum()):>10d}")
    if dry_run:
        return None

    pool = np.where(lab_ok if drop_train else (lab_ok & (d_train >= buffer_m)))[0]
    if len(pool) == 0:
        raise ValueError(f"{name}: no candidate pixel is {buffer_m:.0f} m from every existing point; "
                         "lower buffer_m (see the table above) or use drop_train=True")
    rng.shuffle(pool)
    index, chosen = SpacingIndex(MIN_SAMPLE_DIST_M), []
    for c in pool:
        if len(chosen) >= n_nat:
            break
        x, y = centres[c]
        if not index.too_close(x, y, MIN_SAMPLE_DIST_M):
            index.add(x, y)
            chosen.append(c)
    if len(chosen) < n_nat:
        print(f"  WARNING: only {len(chosen)} natural building points found (wanted {n_nat})")

    bg = background_points(cx, cy, tree, cfg.get("ndsm"), crs, n_bg * 4, rng, boundary)
    if len(bg) and not drop_train:
        bg = bg[ttree.query(bg, p=pnorm)[0] >= buffer_m]
    bg = bg[:n_bg]
    if len(bg) < n_bg:
        print(f"  WARNING: only {len(bg)} natural background points found (wanted {n_bg})")

    rows = []
    for c in chosen:
        zz = float(zbar[c])
        rows.append(dict(x_m=centres[c, 0], y_m=centres[c, 1], height_m=zz, is_background=0,
                         n_parts_50m=int(n_parts[c]), slice_lo=int(min(np.floor(zz), HEIGHT_TOP_M)),
                         split="test"))
    for x, y in bg:
        rows.append(dict(x_m=x, y_m=y, height_m=0.0, is_background=1, n_parts_50m=0,
                         slice_lo=-1, split="test"))
    gdf = gpd.GeoDataFrame(rows, geometry=[Point(r["x_m"], r["y_m"]) for r in rows], crs=crs)
    gdf.insert(0, "city", name)
    gdf.insert(0, "sample_id", [f"{name}_nat_{i:06d}" for i in range(len(gdf))])
    nxy = np.c_[gdf.x_m.to_numpy(), gdf.y_m.to_numpy()]

    out = cfg["output"] + "_natural"
    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)
    if drop_train:
        # training points that sit too close to a test point are flagged, not deleted
        d_to_test, _ = cKDTree(nxy).query(txy, p=pnorm)
        drop = d_to_test < buffer_m
        keep_xy = txy[~drop]
        print(f"  drop_train: {int(drop.sum())} of {len(txy)} training points ({drop.mean():.1%}) are within "
              f"{buffer_m:.0f} m of a test point")
        tm = tmeta.loc[drop, ["sample_id", "height_m", "is_background"]].copy()
        tm["dist_to_test_m"] = d_to_test[drop].round(1)
        tm.to_csv(out + "_drop_train.csv", index=False)
        print("  training points flagged, by class (flagged / all):")
        lab = tmeta.height_m.to_numpy(); isbg = tmeta.is_background.to_numpy() == 1
        print("      bg".ljust(10) + f"{int((drop & isbg).sum()):>8d} / {int(isbg.sum())}")
        for (lo, hi, lbl) in NAT_CLASSES[1:]:
            m = (~isbg) & (lab >= lo) & (lab < hi)
            print(f"    {lbl:>8}".ljust(10) + f"{int((drop & m).sum()):>8d} / {int(m.sum())}")
        ref_tree = cKDTree(keep_xy)
    else:
        ref_tree = ttree
    # distance from each test point to the nearest remaining training point (radius, metres):
    # lets you evaluate the same test set again on only the points >= 100 m away, no new export
    gdf["dist_to_train_m"] = ref_tree.query(nxy)[0].round(1)

    bld = gdf[gdf.is_background == 0]
    cc = _class_counts(bld.height_m.to_numpy())
    pop = _class_counts(zbar[lab_ok])
    print("  natural sample by class (share of sample / share of the city's candidate pixels):")
    for (lo, hi, lbl), n, p in zip(NAT_CLASSES, cc, pop):
        print(f"    {lbl:>8}: {n:5d}  ({n / max(len(bld), 1):6.1%} / {p / max(sum(pop), 1):6.1%})")
    print(f"  {len(bld)} building + {len(gdf) - len(bld)} background test points")

    gdf.to_file(out + ".gpkg", driver="GPKG")
    wgs = gdf.to_crs(4326)
    wgs.to_file(out + ".geojson", driver="GeoJSON")
    csv = pd.DataFrame(wgs.drop(columns="geometry"))
    csv["longitude"], csv["latitude"] = wgs.geometry.x.to_numpy(), wgs.geometry.y.to_numpy()
    csv.to_csv(out + "_wgs84.csv", index=False)
    print(f"  wrote {out}.gpkg / .geojson / _wgs84.csv")
    return gdf


def natural_distances(name, cfg, train_table):
    """Distance from each natural test point to the nearest REMAINING training point (after the
    drop_train flags of the 50 m run are removed), as a radius and as a square (Chebyshev).
    Writes <output>_natural_dist.csv (sample_id, dist_to_train_m, cheb_to_train_m); the draw is untouched.
    For the CNN: report the test error on all points and on cheb_to_train_m >= 330, the points whose
    330 m square holds no training point. The gap between the two is the leakage you are living with."""
    out = cfg["output"] + "_natural"
    nat = pd.read_csv(out + "_wgs84.csv", usecols=["sample_id", "x_m", "y_m"])
    tt = pd.read_csv(train_table, usecols=["city", "sample_id", "x_m", "y_m"])
    tt = tt[tt.city == name]
    drop = pd.read_csv(out + "_drop_train.csv").sample_id
    tt = tt[~tt.sample_id.isin(drop)]
    q = nat[["x_m", "y_m"]].to_numpy(float)
    t = cKDTree(tt[["x_m", "y_m"]].to_numpy(float))
    nat["dist_to_train_m"] = t.query(q)[0].round(1)
    nat["cheb_to_train_m"] = t.query(q, p=np.inf)[0].round(1)
    nat.drop(columns=["x_m", "y_m"]).to_csv(out + "_dist.csv", index=False)
    n = len(nat)
    print(f"{name}: {n} test points; >= 100 m (radius): {int((nat.dist_to_train_m >= 100).sum())}, "
          f"no training point inside a 330 m square: {int((nat.cheb_to_train_m >= 330).sum())}")
    return nat

In [ ]:
for name in ACTIVE_CITIES:
    natural_test_points(name, CITIES[name], train_table=THREE_CITY_TRAIN_ROWS, buffer_m=50, drop_train=True)

## 5. Add the five new cities
Run the three steps in order, setting `STEP` in the config cell each time: `"audit"` (preview, nothing but one CSV is
written), `"train"` (candidate points), then, after section 6, `"natural"`.

In [ ]:
# =====================================================================
# ADD MORE CITIES (Recife, Salvador, Curitiba, Niteroi, Belo Horizonte,
# Florianopolis) to the sample, to get more tall buildings.
# Notebook cell. Runs in the SAME notebook as the sampler and natural_test_cell:
# it reuses CITIES, load_parts, candidate_pixels, area_weighted_height,
# stratified_sample, make_slices, run_city, natural_test_points, inside_boundary
# and the sampler constants. Run those cells first.
#
# STEP = "audit"    reads the footprints of each new city and PREVIEWS the sample
#                   (same rules as the sampler, nothing written except one CSV):
#                   files present, CRS zone, height field, and how many samples
#                   you would get per height class, in particular >= 30 m and the
#                   pooled top slice (>= 49 m). Compared with the cities already
#                   in the merged table. Read this before spending Earth Engine time.
# STEP = "train"    run_city() for each new city: writes <output>.gpkg / .geojson /
#                   _wgs84.csv / _slices.csv, same as for the first three cities.
# STEP = "natural"  natural_test_points(drop_train=True, buffer_m=50) for each new
#                   city, using the new training points as the exclusion reference.
#
# =====================================================================
import os
import numpy as np
import pandas as pd

# ------------------------- CONFIG ------------------------------------
STEP = "train"                                 # "audit" | "train" | "natural"  (run the cell once per step)
NEW_CITIES = ["belo_horizonte", "salvador", "recife", "curitiba","niteroi"]

EXISTING_TABLE = THREE_CITY_TABLE      # the merged table before the rebalancing (reconstructed above)
OUT = OUT_NEW
QUOTA = SAMPLES_PER_SLICE                      # per 1 m slice and city (200 so far); keep equal to the first cities
N_BG = N_BACKGROUND                            # background points per city
NAT_N, NAT_BG = 1000, 100                      # natural test points per city: SET THESE to what you used for the first three
TALL_M = 30.0                                  # "tall" for the preview tables
# ---------------------------------------------------------------------

os.makedirs(OUT, exist_ok=True)
_CLASSES = [(3, 6, "3-6"), (6, 15, "6-15"), (15, 30, "15-30"), (30, 49, "30-49"), (49, np.inf, ">=49")]


def _exists(p):
    return bool(p) and os.path.exists(p)


def _class_row(z):
    z = np.asarray(z)
    return {lbl: int(((z >= lo) & (z < hi)).sum()) for lo, hi, lbl in _CLASSES}


def audit_city(name):
    cfg = CITIES[name]
    row = dict(city=name)
    fp, nd, bd = cfg.get("footprints"), cfg.get("ndsm"), cfg.get("boundary")
    row.update(footprints=_exists(fp), ndsm=(_exists(nd) if nd else None), boundary=(_exists(bd) if "boundary" in cfg else None))
    if not row["footprints"]:
        print(f"{name}: footprints NOT FOUND at {fp}")
        return row
    print(f"{name}:")
    cx, cy, area, z, crs = load_parts(fp, cfg["height_field"], MIN_PART_AREA_M2)
    epsg, want = crs.to_epsg(), cfg.get("gee_epsg")
    row.update(parts=len(cx), epsg=epsg, gee_epsg=want, zone_ok=(epsg is not None and want is not None and epsg % 100 == want % 100))
    if want and not row["zone_ok"]:
        print(f"  WARNING: footprints in EPSG:{epsg}, Earth Engine config EPSG:{want} (different UTM zone)")
    row.update(parts_ge30=int((z >= 30).sum()), parts_ge50=int((z >= 50).sum()), h_p99=float(np.percentile(z, 99)),
               h_max=float(z.max()))
    from scipy.spatial import cKDTree
    tree = cKDTree(np.c_[cx, cy])
    centres = candidate_pixels(cx, cy)
    if "boundary" in cfg and _exists(cfg["boundary"]) and "inside_boundary" in globals():
        centres = centres[inside_boundary(cfg["boundary"], centres, crs)]
        row["boundary_applied"] = True
    zbar, _ = area_weighted_height(centres, tree, z, area, NEIGHBOUR_RADIUS_M)
    row["candidates"] = len(centres)
    row.update({f"cand_{k}": v for k, v in _class_row(zbar).items()})
    rng = np.random.default_rng(SEED)
    slices = make_slices()
    chosen, counts, pop = stratified_sample(zbar, centres, slices, QUOTA, MIN_SAMPLE_DIST_M, rng)
    zs = np.array([zbar[c] for c, _ in chosen])
    row["samples"] = len(chosen)
    row.update({f"samp_{k}": v for k, v in _class_row(zs).items()})
    row["samples_ge30"] = int((zs >= TALL_M).sum())
    row["slices_ge30_below_quota"] = int(sum(1 for (lo, _), c in zip(slices, counts) if lo >= TALL_M and c < QUOTA))
    print(f"  {len(cx)} parts ({row['parts_ge30']} >= 30 m, {row['parts_ge50']} >= 50 m), {len(centres)} candidate pixels, "
          f"preview: {len(chosen)} samples, {row['samples_ge30']} >= {TALL_M:g} m, {row['samp_>=49']} in the top slice")
    return row


def existing_summary():
    if not _exists(EXISTING_TABLE):
        return None
    t = pd.read_csv(EXISTING_TABLE, usecols=["city", "height_m", "is_background", "split"])
    t = t[(t.is_background == 0) & (t.split == "train")]
    rows = []
    for c, g in t.groupby("city"):
        r = dict(city=c, samples=len(g), samples_ge30=int((g.height_m >= TALL_M).sum()))
        r.update({f"samp_{k}": v for k, v in _class_row(g.height_m).items()})
        rows.append(r)
    return pd.DataFrame(rows)


def do_audit():
    rows = [audit_city(n) for n in NEW_CITIES if n in CITIES]
    missing = [n for n in NEW_CITIES if n not in CITIES]
    if missing:
        print(f"Not in CITIES (add them to the notebook config): {missing}")
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT}/new_cities_audit.csv", index=False)
    ok = df[df.footprints == True]
    if len(ok):
        cols = ["city", "parts", "candidates", "samples", "samples_ge30", "samp_30-49", "samp_>=49", "slices_ge30_below_quota"]
        print("\nPreview of the new training samples (building points, stratified, quota %d per slice):" % QUOTA)
        print(ok[cols].to_string(index=False))
        ex = existing_summary()
        new_ge30 = int(ok.samples_ge30.sum())
        if ex is not None and len(ex):
            print("\nAlready in the merged table (split = train, buildings only):")
            print(ex[["city", "samples", "samples_ge30", "samp_30-49", "samp_>=49"]].to_string(index=False))
            old_ge30 = int(ex.samples_ge30.sum())
            print(f"\n>= {TALL_M:g} m samples: {old_ge30} now, {new_ge30} more from the new cities "
                  f"({old_ge30 + new_ge30} in total, x{(old_ge30 + new_ge30) / max(old_ge30, 1):.2f}).")
        else:
            print(f"\n>= {TALL_M:g} m samples from the new cities: {new_ge30} (no merged table found at {EXISTING_TABLE} to compare)")
    # the nDSM is optional (ndsm=None in CITIES): only a configured but missing file counts as a problem
    bad = df[(df.footprints != True) | (df.ndsm == False) | (df.boundary == False)] if len(df) else df
    if len(bad):
        print("\nFILES MISSING:")
        print(bad[["city", "footprints", "ndsm", "boundary"]].to_string(index=False))
    return df


def _ready():
    """New cities that are configured and whose footprint file exists; the others are reported and skipped."""
    ok = []
    for n in NEW_CITIES:
        if n not in CITIES:
            print(f"{n}: not in CITIES, skipped")
        elif not _exists(CITIES[n].get("footprints")):
            print(f"{n}: footprints not found at {CITIES[n].get('footprints')}, skipped")
        else:
            ok.append(n)
    return ok


def do_train():
    for n in _ready():
        run_city(n, CITIES[n], quota=QUOTA, n_bg=N_BG)
    print("\nNext: upload each <output>.geojson (or _wgs84.csv) as an Earth Engine table asset named "
          "<city>_sample_points_selected_wgs84 under projects/YOUR-EARTH-ENGINE-PROJECT/assets/frantz_replication/. Then STEP = 'natural'.")


def _train_ref(cfg):
    """Reference training points for the natural test draw: the rebalanced selection if
    rebalance_training_cell.py has been run (those are the points that get exported), else all candidates."""
    sel = cfg["output"] + "_selected_wgs84.csv"
    ref = sel if os.path.exists(sel) else cfg["output"] + "_wgs84.csv"
    print(f"  training reference: {os.path.basename(ref)}")
    return ref


def do_natural():
    for n in _ready():
        cfg = CITIES[n]
        natural_test_points(n, cfg, n_nat=NAT_N, n_bg=NAT_BG, buffer_m=50.0, drop_train=True,
                            train_table=_train_ref(cfg))
    print("\nNext: upload each <output>_natural.geojson as <city>_sample_points_natural_wgs84, then run the Earth Engine exporter "
          "with POINTS_SET = 'train' and again with 'natural' (ACTIVE_CITIES = the six new cities).")

In [ ]:
do_audit()    # STEP = "audit"

In [ ]:
do_train()    # STEP = "train"

## 6. Rebalance the training set
Writes `unused_existing_ids.csv` (also shipped in `data/`) and the selected points to upload to Earth Engine.

In [ ]:
# =====================================================================
# REBALANCE THE TRAINING SET: more tall buildings, same total size, balanced cities.
# Notebook cell. Run it AFTER add_cities_cell.py STEP = "train" (the new cities' candidate
# points exist as <output>_wgs84.csv) and BEFORE the Earth Engine exports of the new cities.
# Needs CITIES from the sampler notebook (only for the new cities' output paths).
#
# What it does (no model is trained, no feature is read; it works on sample_id, city, height_m,
# is_background, x_m, y_m only):
#   pool    = the current training rows of the old cities (merged table, split == "train")
#             + all candidate points the sampler drew for the new cities
#   target  = N_TARGET building rows (default: the size of the current training set, so the
#             dataset does not grow) and BG_TARGET background rows
#   step 1  how many rows per 1 m height slice: water-filling. Every slice gets the same count T,
#           except slices with fewer than T available points, which give everything they have;
#           T is the largest value that fits the target (the pooled top slice [49, inf) is kept whole). Tall slices are the scarce ones, so they
#           are kept whole and the abundant short slices are thinned.
#   step 2  inside each slice, how many per city: the same water-filling over cities, so each
#           city gets an equal share where it can and every city is represented in short buildings
#   step 3  within a city and slice, rows are picked at random (seeded). All points already respect
#           the 50 m spacing, which the sampler enforced inside each city.
# Slices follow the sampler: [3,4), ..., [48,49), and [49, inf) pooled.
#
# Writes (OUT):
#   selection_all.csv              every pool row with `selected`
#   unused_existing_ids.csv        old-city rows that are not kept -> merge_gee_exports.py --unused-ids
#   rebalance_summary_*.csv        counts per city x class and per slice, before and after
#   rebalance_before_after.png     per-slice counts by city: current set, available pool, new set
# and, for each new city, next to the sampler files:
#   <output>_selected_wgs84.csv / <output>_selected.geojson   upload THIS as the Earth Engine asset
#                                                             <city>_sample_points_selected_wgs84
# Nothing is deleted or overwritten.
# =====================================================================
import os
import numpy as np
import pandas as pd

# ------------------------- CONFIG ------------------------------------
OUT = OUT_NEW
OLD_CITIES = ["porto_alegre", "sao_paulo", "rio_de_janeiro"]
NEW_CITIES = ["belo_horizonte", "salvador", "recife", "curitiba", "niteroi"]
N_TARGET = None          # building rows in the new training set; None = same as the current set
SWEEP = [6000, 8000, 10000, 12000, 14000]   # also prints what other target sizes would give (no file is written for them)
BG_TARGET = None         # background rows; None = same as the current set
SEED = 42
KEEP_TOP_WHOLE = True    # the pooled top slice [49 m, inf) is kept whole: it covers many 1 m slices and is the rarest class
TOP_SLICE = 49           # last slice [49, inf), as in the sampler
# ---------------------------------------------------------------------

CITY_ORDER = OLD_CITIES + NEW_CITIES
CLASSES = [(3, 6, "3-6 m"), (6, 15, "6-15 m"), (15, 30, "15-30 m"), (30, 49, "30-49 m"), (49, np.inf, ">=49 m")]
COLS = ["sample_id", "city", "height_m", "is_background", "x_m", "y_m"]
os.makedirs(OUT, exist_ok=True)


def waterfill(avail, budget, rng):
    """Integer allocation x with x_i <= avail_i, sum x = min(budget, sum avail), as equal as possible:
    x_i = min(avail_i, T) for the largest T that fits, then the remainder goes +1 to random items above T."""
    avail = np.asarray(avail, dtype=int)
    if budget >= avail.sum():
        return avail.copy()
    lo, hi = 0, int(avail.max())
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if np.minimum(avail, mid).sum() <= budget:
            lo = mid
        else:
            hi = mid - 1
    x = np.minimum(avail, lo)
    rem = int(budget - x.sum())
    if rem:
        x[rng.choice(np.where(avail > lo)[0], size=rem, replace=False)] += 1
    return x


def slice_of(h):
    return np.minimum(np.floor(np.asarray(h, float)), TOP_SLICE).astype(int)


def load_pool():
    t = pd.read_csv(EXISTING_TABLE, usecols=COLS + ["split"])
    old = t[(t.split == "train") & t.city.isin(OLD_CITIES)][COLS].assign(source="existing")
    frames = [old]
    for n in NEW_CITIES:
        p = CITIES[n]["output"] + "_wgs84.csv"
        if not os.path.exists(p):
            print(f"{n}: {p} not found (run add_cities_cell.py with STEP = 'train'), skipped")
            continue
        d = pd.read_csv(p)
        d["city"] = n
        frames.append(d[COLS].assign(source="new"))
    pool = pd.concat(frames, ignore_index=True)
    pool["is_background"] = pool.is_background.astype(int)
    pool["slice"] = np.where(pool.is_background == 1, -1, slice_of(pool.height_m))
    if not pool.sample_id.is_unique:
        raise SystemExit("sample_id is not unique across the pool")
    return pool, old


def slice_targets(avail, slices, n_target, rng):
    """Step 1: rows per slice. The pooled top slice is exempt from the per-slice cap when KEEP_TOP_WHOLE."""
    avail = np.asarray(avail, int)
    if KEEP_TOP_WHOLE and TOP_SLICE in slices:
        i = list(slices).index(TOP_SLICE)
        t = np.zeros(len(avail), int)
        t[i] = avail[i]
        rest = np.array([k for k in range(len(avail)) if k != i])
        t[rest] = waterfill(avail[rest], max(n_target - avail[i], 0), rng)
        return t
    return waterfill(avail, n_target, rng)


def balance(pool, n_target, bg_target, seed=SEED):
    rng = np.random.default_rng(seed)
    cities = [c for c in CITY_ORDER if c in set(pool.city)]
    sel = np.zeros(len(pool), bool)
    bld = pool[pool.is_background == 0]
    slices = sorted(bld["slice"].unique())
    avail_s = np.array([(bld["slice"] == s).sum() for s in slices])
    t_s = slice_targets(avail_s, slices, n_target, rng)                        # step 1
    for s, ts in zip(slices, t_s):
        g = bld[bld["slice"] == s]
        a_c = np.array([(g.city == c).sum() for c in cities])
        t_c = waterfill(a_c, ts, rng)                                         # step 2
        for c, k in zip(cities, t_c):
            if k:
                sel[rng.choice(g.index[g.city == c].to_numpy(), size=k, replace=False)] = True   # step 3
    bg = pool[pool.is_background == 1]
    a_c = np.array([(bg.city == c).sum() for c in cities])
    for c, k in zip(cities, waterfill(a_c, bg_target, rng)):
        if k:
            sel[rng.choice(bg.index[bg.city == c].to_numpy(), size=k, replace=False)] = True
    return sel


def class_table(df):
    b = df[df.is_background == 0]
    rows = {c: [int(((b.city == c) & (b.height_m >= lo) & (b.height_m < hi)).sum()) for lo, hi, _ in CLASSES] for c in CITY_ORDER}
    t = pd.DataFrame(rows, index=[n for _, _, n in CLASSES]).T
    t["total"] = t.sum(axis=1)
    t.loc["ALL"] = t.sum()
    bg = df[df.is_background == 1].groupby("city").size()
    t["background"] = [int(bg.get(c, 0)) for c in CITY_ORDER] + [int(bg.sum())]
    return t[t.total + t.background > 0]


def sweep(pool, current_n):
    """Step 1 only (rows per slice) for several target sizes: how flat can the set be, and how big must it be?"""
    bld = pool[pool.is_background == 0]
    slices = sorted(bld["slice"].unique())
    avail = np.array([(bld["slice"] == s).sum() for s in slices])
    rng = np.random.default_rng(SEED)
    rows = []
    for n in sorted(set(SWEEP + [current_n])):
        t = slice_targets(avail, slices, n, rng)
        cap = int(np.max(np.where(t < avail, t, 0))) if (t < avail).any() else int(t.max())
        cls = lambda lo, hi: int(sum(x for s, x in zip(slices, t) if lo <= s < hi))
        tall = cls(30, 10**6)
        rows.append(dict(target_rows=n, rows_per_slice_cap=cap, rows_3_15=cls(3, 15), rows_15_30=cls(15, 30), rows_30_plus=tall,
                         share_30_plus=f"{tall / max(int(t.sum()), 1):.0%}", slices_taken_whole=int((t == avail).sum())))
    print("What other target sizes would give (step 1 only; the current training-set size is the row marked *):")
    out = pd.DataFrame(rows)
    out["target_rows"] = [f"{n}{'*' if n == current_n else ''}" for n in out.target_rows]
    print(out.to_string(index=False)); print()


def rebalance():
    pool, old = load_pool()
    n_target = N_TARGET or int((old.is_background == 0).sum())
    bg_target = BG_TARGET if BG_TARGET is not None else int((old.is_background == 1).sum())
    sweep(pool, n_target)
    sel = balance(pool, n_target, bg_target)
    pool["selected"] = sel
    pool.drop(columns="slice").to_csv(f"{OUT}/selection_all.csv", index=False)
    keep = pool[pool.selected]
    cur = pool[pool.source == "existing"]

    print(f"Pool: {int((pool.is_background == 0).sum())} building rows ({len(cur)} current + new candidates). "
          f"Target: {n_target} building + {bg_target} background rows.\n")
    tb, ta, tp = class_table(cur), class_table(keep), class_table(pool)
    for name, t in [("CURRENT training set", tb), ("AVAILABLE pool (current + new candidates)", tp), ("NEW training set", ta)]:
        print(name); print(t.to_string()); print()
        t.to_csv(f"{OUT}/rebalance_summary_{name.split()[0].lower()}_by_class.csv")

    sl = (pd.DataFrame({"current": cur[cur.is_background == 0].groupby("slice").size(),
                        "available": pool[pool.is_background == 0].groupby("slice").size(),
                        "selected": keep[keep.is_background == 0].groupby("slice").size()}).fillna(0).astype(int))
    sl.to_csv(f"{OUT}/rebalance_summary_by_slice.csv")
    tall = lambda d: int(((d.is_background == 0) & (d.height_m >= 30)).sum())
    print(f">= 30 m rows: {tall(cur)} now -> {tall(keep)} after. Rows per slice: current {int(sl.current.min())}-{int(sl.current.max())}, "
          f"new {int(sl.selected.min())}-{int(sl.selected.max())}.")
    print("Rows per city, building + background (current -> new): " +
          ", ".join(f"{c} {int((cur.city == c).sum())} -> {int((keep.city == c).sum())}" for c in CITY_ORDER if (pool.city == c).any()))

    # files for the next steps
    unused = pool[(pool.source == "existing") & ~pool.selected]
    unused[["sample_id"]].to_csv(f"{OUT}/unused_existing_ids.csv", index=False)
    print(f"\n{len(unused)} current training rows are not kept -> {OUT}/unused_existing_ids.csv "
          f"(merge_gee_exports.py --unused-ids; they stay in the table as split=unused)")
    import geopandas as gpd
    for n in NEW_CITIES:
        p = CITIES[n]["output"] + "_wgs84.csv"
        if not os.path.exists(p):
            continue
        full = pd.read_csv(p)
        ids = set(keep.sample_id[keep.city == n])
        sub = full[full.sample_id.isin(ids)].reset_index(drop=True)
        base = CITIES[n]["output"]
        sub.to_csv(base + "_selected_wgs84.csv", index=False)
        gpd.GeoDataFrame(sub, geometry=gpd.points_from_xy(sub.longitude, sub.latitude), crs=4326).to_file(
            base + "_selected.geojson", driver="GeoJSON")
        print(f"  {n}: {len(sub)} of {len(full)} candidate points kept -> {base}_selected.geojson / _selected_wgs84.csv")
    make_chart(cur, pool, keep)
    return pool


def make_chart(cur, pool, keep):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#d9d8d3"
    PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]   # fixed order, one hue per city
    cities = [c for c in CITY_ORDER if (pool.city == c).any()]
    colour = {c: PAL[i % len(PAL)] for i, c in enumerate(CITY_ORDER)}
    xs = list(range(3, TOP_SLICE + 1))
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, facecolor=SURF)
    for ax, (title, d) in zip(axes, [("Current training set", cur), ("Available pool (current + new candidates)", pool),
                                     ("New training set (same size, balanced)", keep)]):
        b = d[d.is_background == 0]
        bottom = np.zeros(len(xs))
        for c in cities:
            cnt = np.array([int(((b.city == c) & (b["slice"] == s)).sum()) for s in xs])
            ax.bar(xs, cnt, bottom=bottom, color=colour[c], edgecolor=SURF, linewidth=0.8, width=0.9)
            bottom += cnt
        ax.set_facecolor(SURF); ax.set_title(title, loc="left", fontsize=11, color=INK, weight="bold")
        ax.set_ylabel("rows per 1 m slice", color=INK2); ax.tick_params(colors=INK2)
        ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
        for sp in ("top", "right", "left"):
            ax.spines[sp].set_visible(False)
        ax.spines["bottom"].set_color(GRID)
        ax.text(0.995, 0.93, f"{int(b.shape[0])} rows, {int((b.height_m >= 30).sum())} at 30 m or taller", transform=ax.transAxes,
                ha="right", va="top", fontsize=9, color=INK2)
    axes[-1].set_xlabel("building height slice (m; the last slice pools everything from 49 m)", color=INK2)
    axes[0].legend(handles=[Patch(fc=colour[c], ec=SURF, label=c.replace("_", " ").title()) for c in cities], ncol=len(cities),
                   frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.12), fontsize=9, labelcolor=INK2)
    fig.tight_layout()
    fig.savefig(f"{OUT}/rebalance_before_after.png", dpi=160, facecolor=SURF)
    print(f"chart: {OUT}/rebalance_before_after.png")


pool_after = rebalance()

## 7. Natural test points of the five new cities
Run after the rebalancing, so the exclusion reference is the selected training set.

In [ ]:
do_natural()    # STEP = "natural"

## 8. Which statistic is `height_m`? (check, optional)
For a sample of footprint parts: mean, median, p90 or max of the nDSM inside the footprint, per city.

In [ ]:
import os
# =====================================================================
# WHICH STATISTIC IS height_m?  Notebook cell, same notebook as the sampler (reuses CITIES).
# The nDSM rasters exist for the new cities, so the statistic can be recovered from the data:
# for a random sample of footprint parts we compute mean / median / p90 / max of the nDSM
# inside each footprint and see which one height_m matches, per city. If the best statistic
# differs between cities, height_m is not comparable across cities and the labels need rebuilding.
# Light: reads only the pixels under each sampled footprint (windowed), N_PARTS per city.
# =====================================================================
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.mask import mask as rio_mask
from scipy.stats import spearmanr

CHECK_CITIES = list(CITIES)          # or a subset, e.g. ["belo_horizonte", "salvador", "niteroi", "rio_de_janeiro"]
N_PARTS = 2000                       # random parts per city
N_TALL = 1000                        # extra parts with height_m >= TALL_M, because random parts are almost all low
TALL_M = 20.0
MIN_AREA = 30.0                      # m2, skip tiny parts (few pixels)
OUT_CSV = os.path.join(config.out("new_cities"), "height_statistic_check.csv")
STATS = {"mean": np.mean, "median": np.median, "p90": lambda v: np.percentile(v, 90), "max": np.max}


def check_city(name, rng):
    cfg = CITIES[name]
    if not (cfg.get("ndsm") and os.path.exists(cfg["ndsm"])):
        print(f"{name}: no nDSM file, skipped"); return None
    g = gpd.read_file(cfg["footprints"])
    with rasterio.open(cfg["ndsm"]) as _src:
        g = g.to_crs(_src.crs)               # first: the area filter below needs metres, and Recife's footprints are in EPSG:4326
    g = g[g.geometry.notna() & ~g.geometry.is_empty].explode(index_parts=False)
    g = g[g.geometry.geom_type == "Polygon"]
    h = pd.to_numeric(g[cfg["height_field"]], errors="coerce")
    g = g[(h > 0) & (g.geometry.area >= MIN_AREA)]
    hh = pd.to_numeric(g[cfg["height_field"]])
    tall = g[hh >= TALL_M]
    g = pd.concat([g.sample(min(N_PARTS, len(g)), random_state=int(rng.integers(1e9))),
                   tall.sample(min(N_TALL, len(tall)), random_state=int(rng.integers(1e9)))]).loc[lambda x: ~x.index.duplicated()]
    rows = []
    with rasterio.open(cfg["ndsm"]) as src:
        for geom, hz in zip(g.geometry, pd.to_numeric(g[cfg["height_field"]])):
            try:
                arr, _ = rio_mask(src, [geom], crop=True, filled=False, all_touched=False)
            except ValueError:
                continue                      # footprint outside the raster
            v = np.ma.compressed(arr[0]).astype(float)
            v = v[np.isfinite(v)]
            if src.nodata is not None:
                v = v[v != src.nodata]
            if len(v) >= 3:
                rows.append(dict(h=hz, **{k: f(v) for k, f in STATS.items()}))
    d = pd.DataFrame(rows)
    if len(d) < 50:
        print(f"{name}: only {len(d)} footprints overlap the nDSM, skipped"); return None
    res = dict(city=name, n=len(d), h_median=d.h.median(), share_3_6=float(((d.h >= 3) & (d.h < 6)).mean()))
    for k in STATS:
        ok = d[k] > 0.5
        res[f"{k}_ratio"] = float(np.median(d.h[ok] / d[k][ok]))            # 1.0 = height_m equals this statistic
        res[f"{k}_mae"] = float(np.median(np.abs(d.h - d[k])))
        res[f"{k}_rho"] = float(spearmanr(d.h, d[k])[0])
    res["best_by_mae"] = min(STATS, key=lambda k: res[f"{k}_mae"])
    # does the ratio depend on height? median of height_m / nDSM median, per height_m class
    for lo, hi, lbl in [(3, 6, "3-6"), (6, 15, "6-15"), (15, 30, "15-30"), (30, 1e9, "30+")]:
        m = (d.h >= lo) & (d.h < hi) & (d["median"] > 0.5)
        res[f"ratio_med_{lbl}"] = float(np.median(d.h[m] / d["median"][m])) if m.sum() >= 10 else np.nan
        res[f"n_{lbl}"] = int(m.sum())
    return res


rng = np.random.default_rng(0)
out = [r for r in (check_city(n, rng) for n in CHECK_CITIES if n in CITIES) if r]
tab = pd.DataFrame(out)
tab.to_csv(OUT_CSV, index=False)
cols = ["city", "n", "h_median", "share_3_6", "best_by_mae"] + [f"{k}_{m}" for k in STATS for m in ("ratio", "mae")]
print(tab[cols].round(2).to_string(index=False))
print("\nheight_m / nDSM median by height_m class (1.0 = same as the median):")
print(tab[["city"] + [f"ratio_med_{l}" for l in ("3-6", "6-15", "15-30", "30+")] + [f"n_{l}" for l in ("3-6", "6-15", "15-30", "30+")]].round(2).to_string(index=False))
print("\nReading: <stat>_ratio close to 1 and a small <stat>_mae mean height_m IS that statistic. "
      "If best_by_mae differs between cities, the labels are not comparable.")